Telco Customer Churn dataset

In [9]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score
import joblib

print("Libraries successfully import ho gayi hain!")

Libraries successfully import ho gayi hain!


In [10]:
# 1. Dataset load karein (Make sure file aapke same folder mein ho)
df = pd.read_csv('/content/Telco-Customer-Churn.csv')

# 2. TotalCharges column ko numeric mein badlein aur missing values ko median se fill karein
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df['TotalCharges'] = df['TotalCharges'].fillna(df['TotalCharges'].median())

# 3. Target variable 'Churn' ko 0 aur 1 mein convert karein
df['Churn'] = df['Churn'].apply(lambda x: 1 if x == 'Yes' else 0)

print("Data successfully load aur clean ho gaya hai. Data ki shape:", df.shape)

Data successfully load aur clean ho gaya hai. Data ki shape: (7043, 21)


In [11]:
# Target 'Churn' aur 'customerID' ko drop karke baki sab X mein dalein
X = df.drop(columns=['customerID', 'Churn'])
y = df['Churn']

# Numerical aur Categorical columns ki list alag karein
numerical_features = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_features = X.select_dtypes(include=['object']).columns.tolist()

print("Numerical Features:", numerical_features)
print("Categorical Features:", categorical_features)

Numerical Features: ['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges']
Categorical Features: ['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod']


In [12]:
# Numerical columns ke liye transformer
numerical_transformer = Pipeline(steps=[
    ('scaler', StandardScaler())
])

# Categorical columns ke liye transformer
categorical_transformer = Pipeline(steps=[
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])

# Dono ko combine karke final preprocessor banayein
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numerical_transformer, numerical_features),
        ('cat', categorical_transformer, categorical_features)
    ])

print("Preprocessing pipeline tayar hai!")

Preprocessing pipeline tayar hai!


In [13]:
# Final Pipeline: Pehle data preprocess hoga, phir model train hoga
final_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(random_state=42))
])

# Data ko 80% Training aur 20% Testing mein divide karein
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print("Data split ho gaya aur final pipeline structure tayar hai.")

Data split ho gaya aur final pipeline structure tayar hai.


In [14]:
# Tuning ke liye parameters set karein
param_grid = {
    'classifier__n_estimators': [50, 100],
    'classifier__max_depth': [5, 10, None]
}

print("GridSearchCV training shuru ho rahi hai... (Please wait)")
grid_search = GridSearchCV(final_pipeline, param_grid, cv=3, scoring='accuracy', n_jobs=-1)
grid_search.fit(X_train, y_train)

print("Tuning mukammal ho gayi!")
print(f"Sabse behtar parameters: {grid_search.best_params_}")

GridSearchCV training shuru ho rahi hai... (Please wait)
Tuning mukammal ho gayi!
Sabse behtar parameters: {'classifier__max_depth': 10, 'classifier__n_estimators': 100}


In [15]:
# Best model ko nikalien
best_model = grid_search.best_estimator_

# Test data par predict karein
y_pred = best_model.predict(X_test)

# Report print karein
print("\n=== MODEL PERFORMANCE REPORT ===")
print(f"Accuracy Score: {accuracy_score(y_test, y_pred):.4f}")
print("\nClassification Report:\n", classification_report(y_test, y_pred))


=== MODEL PERFORMANCE REPORT ===
Accuracy Score: 0.8027

Classification Report:
               precision    recall  f1-score   support

           0       0.84      0.90      0.87      1035
           1       0.66      0.53      0.59       374

    accuracy                           0.80      1409
   macro avg       0.75      0.72      0.73      1409
weighted avg       0.79      0.80      0.80      1409



In [16]:
# Model ko file mein save karein
joblib.dump(best_model, 'customer_churn_pipeline.joblib')

print("Mubarak ho! Aapki complete pipeline 'customer_churn_pipeline.joblib' ke naam se save ho chuki hai.")

Mubarak ho! Aapki complete pipeline 'customer_churn_pipeline.joblib' ke naam se save ho chuki hai.
